# Umbrella Sampling and WHAM PMF Reconstruction

This notebook ties together biased umbrella windows, WHAM reconstruction, and PMF visualization. It is designed to make the connection between restrained sampling and an unbiased free-energy profile explicit.

## WHAM Equations

Umbrella sampling introduces a bias potential in each window, typically

$$U_i^{\mathrm{bias}}(\xi) = \frac{1}{2}k_i(\xi - \xi_i)^2.$$

WHAM then estimates the unbiased probability density $P(\xi)$ self-consistently by combining the window histograms and free-energy offsets. The PMF is reconstructed as

$$G(\xi) = -k_B T\ln P(\xi) + C,$$

where $C$ is an arbitrary additive constant chosen for convenience.

In [ ]:
# Resolve project root and import umbrella sampling and WHAM modules.
from pathlib import Path
import sys
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR, UmbrellaConfig, WHAMConfig
from src.simulate.umbrella import generate_window_centers, run_umbrella_campaign
from src.analyze.wham import solve_wham, bootstrap_pmf_uncertainty
from src.visualization.plot_pmf import plot_pmf

In [ ]:
# Define umbrella sampling parameters and compute evenly spaced window centers.
umbrella_config = UmbrellaConfig(
    xi_min_nm=1.8,
    xi_max_nm=2.6,
    window_spacing_nm=0.1,
    spring_constant_kj_mol_nm2=1000.0,
    per_window_duration_ns=0.5,
    save_interval_ps=2.0,
)
window_centers = generate_window_centers(umbrella_config)
window_centers

In [ ]:
equilibrated_state_path = DATA_DIR / 'trajectories' / 'equilibration' / 'npt_demo' / 'npt_final_state.xml'
system_xml_path = DATA_DIR / 'topologies' / 'example_system.xml'
pull_group_1 = [0, 1, 2]
pull_group_2 = [10, 11, 12]
umbrella_output_dir = DATA_DIR / 'trajectories' / 'umbrella' / 'demo'

# Heavy calculation cell: execute only when the serialized system and equilibrated state are present.
# umbrella_results = run_umbrella_campaign(
#     equilibrated_state_path,
#     system_xml_path,
#     umbrella_config,
#     pull_group_1,
#     pull_group_2,
#     umbrella_output_dir,
# )

In [ ]:
# Generate synthetic xi timeseries, solve WHAM, and plot the PMF with uncertainty.
rng = np.random.default_rng(7)
demo_xi_timeseries = [rng.normal(loc=center, scale=0.05, size=500) for center in window_centers]
spring_constants = np.full(window_centers.size, umbrella_config.spring_constant_kj_mol_nm2, dtype=float)
wham_config = WHAMConfig(tolerance=1e-6, max_iterations=10000, n_bootstrap=16, histogram_bins=80)
wham_result = solve_wham(demo_xi_timeseries, window_centers, spring_constants, 310.0, wham_config)
bootstrap_result = bootstrap_pmf_uncertainty(demo_xi_timeseries, window_centers, spring_constants, 310.0, wham_config)
pmf_figure = plot_pmf(wham_result['xi_bins'], wham_result['pmf_kcal_mol'], bootstrap_result['pmf_std'] / 4.184)
wham_result['converged'], pmf_figure

## Sampling Quality

Window overlap is not optional. If neighboring umbrellas do not overlap in $\xi$, WHAM may still produce a curve, but the result is not trustworthy. Always inspect the histogram support and compare bootstrap uncertainty against the apparent energetic features of the PMF.